
# **Предобработка: Картинки и Данные (OpenCV, PIL, torchvision, pandas)Быстрая загрузка и конвертация (Без ошибок с цветами):**


In [ ]:
import cv2
from PIL import Image
import numpy as np

# OpenCV: Читаем и сразу чиним цвета (BGR -> RGB)
img_bgr = cv2.imread("data/image.jpg")
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# PIL: Загрузка с гарантированным RGB (спасает от ч/б и PNG с альфа-каналом)
img_pil = Image.open("data/image.jpg").convert("RGB")

# **Синхронные аугментации (Картинка + Маска):**

In [ ]:
import torch
from torchvision.transforms import v2

transform = v2.Compose([
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomRotation(degrees=15),
    v2.ColorJitter(brightness=0.2), # Применится только к картинке
    v2.ToImage(), 
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
# img_tensor, mask_tensor = transform(img_pil, mask_pil)

# **Работа с таблицами (Pandas):**


In [ ]:
import pandas as pd

df = pd.read_csv("train.csv")
# Создаем словарь: {путь_к_файлу: метка} для быстрого доступа в Dataset
img_to_label = dict(zip(df['image_path'], df['label']))


# **2. 🧠 Извлечение фичей и Zero-Shot (timm, open_clip, torch)Timm: Достаем эмбеддинги (без классификации):**#


In [ ]:
import timm
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# num_classes=0 сносит "голову", модель возвращает векторы
model_cv = timm.create_model('resnet34', pretrained=True, num_classes=0).to(device)
model_cv.eval()

# На входе тензор [Batch, 3, 224, 224], на выходе [Batch, 512]
with torch.no_grad():
    features = model_cv(image_tensor.to(device))

# **OpenCLIP: Классификация текстом (Zero-Shot):**# 

In [ ]:
import open_clip
import torch.nn.functional as F

model_clip, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
model_clip.eval().to(device)
tokenizer = open_clip.get_tokenizer('ViT-B-32')

# Картинка и кандидаты
image_input = preprocess(Image.open("img.jpg")).unsqueeze(0).to(device)
text_inputs = tokenizer(["a healthy brain", "a brain with a tumor"]).to(device)

with torch.no_grad(), torch.cuda.amp.autocast():
    image_features = model_clip.encode_image(image_input)
    text_features = model_clip.encode_text(text_inputs)
    
    # Нормализация (защита от NaN)
    image_features = F.normalize(image_features, p=2, dim=-1)
    text_features = F.normalize(text_features, p=2, dim=-1)
    
    # Считаем вероятности
    probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)


# **3. 📝 NLP и Генерация (transformers)Правильный промпт через Chat Template (Для LLM):**


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen1.5-1.8B-Chat"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model_llm = AutoModelForCausalLM.from_pretrained(model_id).to(device).eval()

messages = [
    {"role": "system", "content": "You are a helpful AI. Output only JSON."},
    {"role": "user", "content": "Analyze this data..."}
]

# Токенизируем с шаблоном
text_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text_prompt, return_tensors="pt").to(device)

# Генерация
with torch.no_grad():
    output_ids = model_llm.generate(
        **inputs, 
        max_new_tokens=100, 
        temperature=0.1, # 0.1 для точных задач
        do_sample=True
    )

# Обрезаем промпт из ответа
response = tokenizer.decode(output_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)


# **4. 🚀 Классический ML (План "Б"): xgboost, scikit-learnИспользуй это, когда достал векторы (эмбеддинги) из картинок или текста, и нужно быстро обучить классификатор на нестандартных данных без переобучения тяжелой нейронки.**


In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression

# X - матрица фичей numpy [N_samples, 512], y - метки [N_samples]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# XGBoost (Мощно, если данных хотя бы пару тысяч)
xgb_model = xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_val)

# Логистическая регрессия (Идеально для эмбеддингов, если мало данных!)
log_reg = LogisticRegression(max_iter=1000, C=1.0)
log_reg.fit(X_train, y_train)
lr_preds = log_reg.predict(X_val)

print(f"XGB F1: {f1_score(y_val, xgb_preds, average='macro')}")


# **5. ⚙️ База PyTorch: Custom Dataset и Циклы (torch, tqdm)Скелет кастомного датасета (Основа олимпиад):**


In [ ]:
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

class OlympiadDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image) # Для v2 можно передать и маску
            
        return image, torch.tensor(label, dtype=torch.long)

# Использование
dataset = OlympiadDataset(paths_list, labels_list, transform=transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=2)

# Красивый цикл инференса/извлечения с прогресс-баром
all_features = []
for images, _ in tqdm(dataloader, desc="Processing..."):
    images = images.to(device)
    with torch.no_grad():
        features = model_cv(images)
        all_features.append(features.cpu().numpy())

# Собираем все батчи в одну огромную матрицу для XGBoost
X_final = np.vstack(all_features)

# # **NLP**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from torch.optim import AdamW
from tqdm import tqdm
import numpy as np

# ==========================================
# 1. КОНТРАСТИВНЫЙ ЛОСС (NT-Xent)
# ==========================================
class NTXentLoss(nn.Module):
    def __init__(self, temperature=0.05):
        super().__init__()
        self.temperature = temperature
        self.cross_entropy = nn.CrossEntropyLoss()

    def forward(self, embeddings_a, embeddings_b):
        # Матрица косинусных расстояний
        scores = torch.matmul(embeddings_a, embeddings_b.T) / self.temperature
        
        batch_size = embeddings_a.size(0)
        labels = torch.arange(batch_size, device=embeddings_a.device)
        
        # Симметричный лосс (a->b и b->a)
        loss_a = self.cross_entropy(scores, labels)
        loss_b = self.cross_entropy(scores.T, labels)
        
        return (loss_a + loss_b) / 2

# ==========================================
# 2. ДАТАСЕТ И ПУЛИНГ
# ==========================================
class ContrastiveTextDataset(Dataset):
    def __init__(self, texts_a, texts_b, tokenizer, max_len=128):
        self.texts_a = texts_a
        self.texts_b = texts_b
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts_a)

    def tokenize(self, text):
        return self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

    def __getitem__(self, idx):
        tokens_a = self.tokenize(self.texts_a[idx])
        tokens_b = self.tokenize(self.texts_b[idx])
        
        return {
            "input_ids_a": tokens_a["input_ids"].squeeze(0),
            "mask_a": tokens_a["attention_mask"].squeeze(0),
            "input_ids_b": tokens_b["input_ids"].squeeze(0),
            "mask_b": tokens_b["attention_mask"].squeeze(0),
        }

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    return sum_embeddings / sum_mask

# ==========================================
# 3. ОСНОВНОЙ ЦИКЛ (TRAIN LOOP)
# ==========================================
def train_model():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Используем устройство: {device}")

    # Фейковые данные для примера (ЗАМЕНИ НА СВОИ!)
    # anchors - исходные тексты, positives - тексты, похожие по смыслу
    anchors = ["This is a great movie.", "I love learning PyTorch.", "The weather is bad."] * 100
    positives = ["An awesome film.", "Deep learning in PyTorch is fun.", "It is raining heavily."] * 100

    # Настройки
    model_name = "bert-base-uncased"
    batch_size = 16
    epochs = 3
    lr = 2e-5

    # Инициализация
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    
    dataset = ContrastiveTextDataset(anchors, positives, tokenizer)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # Оптимизатор, Лосс, Скейлер (AMP) и Шедулер
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    criterion = NTXentLoss(temperature=0.05)
    scaler = GradScaler()

    total_steps = len(dataloader) * epochs
    warmup_steps = int(total_steps * 0.1)
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

    # Обучение
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")
        
        for batch in progress_bar:
            optimizer.zero_grad()
            
            input_ids_a = batch["input_ids_a"].to(device)
            mask_a = batch["mask_a"].to(device)
            input_ids_b = batch["input_ids_b"].to(device)
            mask_b = batch["mask_b"].to(device)

            # Смешанная точность для скорости и экономии памяти
            with autocast():
                out_a = model(input_ids=input_ids_a, attention_mask=mask_a)
                out_b = model(input_ids=input_ids_b, attention_mask=mask_b)
                
                # Пулинг и нормализация! (Обязательно для косинусного расстояния)
                emb_a = F.normalize(mean_pooling(out_a, mask_a), p=2, dim=1)
                emb_b = F.normalize(mean_pooling(out_b, mask_b), p=2, dim=1)
                
                loss = criterion(emb_a, emb_b)

            # Масштабируем лосс
            scaler.scale(loss).backward()

            # Отменяем масштабирование перед клиппингом (Защита от NaN)
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            # Шаг оптимизатора и обновление скейлера
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            epoch_loss += loss.item()
            progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})
            
        print(f"Средний лосс за эпоху {epoch+1}: {epoch_loss / len(dataloader):.4f}")

    return model, tokenizer

# ==========================================
# 4. ИНФЕРЕНС (ПОЛУЧЕНИЕ ВЕКТОРОВ)
# ==========================================
def get_embeddings(texts, model, tokenizer, device, batch_size=32):
    model.eval()
    all_embeddings = []
    
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
            
            with autocast():
                outputs = model(**inputs)
                embeddings = mean_pooling(outputs, inputs["attention_mask"])
                embeddings = F.normalize(embeddings, p=2, dim=1)
                
            all_embeddings.append(embeddings.cpu().numpy())
            
    return np.vstack(all_embeddings)

# ==========================================
# ЗАПУСК
# ==========================================
if __name__ == "__main__":
    # 1. Запускаем файн-тюнинг
    trained_model, trained_tokenizer = train_model()
    
    # 2. Проверяем инференс на новых данных
    test_texts = ["How to learn deep learning?", "PyTorch vs TensorFlow"]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    test_embs = get_embeddings(test_texts, trained_model, trained_tokenizer, device)
    
    print("\nИнференс завершен!")
    print(f"Размерность полученных фичей: {test_embs.shape}") 
    # Готовые фичи можно закинуть в XGBoost или логистическую регрессию!


# Трик 1: Gradient Accumulation (Градиентное накопление)
Зачем: Тебе нужно обучить тяжелую модель (ViT-Large), но на видеокарте хватает памяти только на batch_size = 4. А для стабильного обучения нужен батч хотя бы 32.Суть: Мы делаем проходы forward и backward маленькими батчами, накапливаем их градиенты, и делаем шаг оптимизатора только раз в несколько итераций.


In [ ]:
# Мы хотим эффективный батч 32, но в память влезает только 4.
# Значит, накапливаем градиенты 8 шагов (4 * 8 = 32)
accumulation_steps = 8 

model.train()
optimizer.zero_grad() # Обнуляем ДО цикла!

for i, (inputs, labels) in enumerate(dataloader):
    inputs, labels = inputs.to(device), labels.to(device)
    
    outputs = model(inputs)
    loss = criterion(outputs, labels)
    
    # КРИТИЧНО: Делим лосс на количество шагов накопления, 
    # чтобы масштаб градиентов соответствовал большому батчу
    loss = loss / accumulation_steps
    loss.backward() # Градиенты суммируются (накапливаются) под капотом
    
    # Делаем шаг оптимизатора только каждый 8-й раз
    if (i + 1) % accumulation_steps == 0:
        optimizer.step()
        optimizer.zero_grad() # Обнуляем только после шага!
        # scheduler.step() # Если используешь шедулер, шаг тоже тут

# **Трик 2: Mixup (Аугментация на уровне тензоров)**

Зачем: Это спасение для Computer Vision, когда мало данных. Нейросеть учится не на чистых картинках, а на "призраках" — полупрозрачных смесях двух картинок.
Суть: Берем картинку кота и картинку собаки. Смешиваем их пиксели (например, 70% кота и 30% собаки). Модель должна предсказать не один класс, а сказать: "Тут 0.7 кота и 0.3 собаки". Это дает невероятную устойчивость (регуляризацию).

In [ ]:
import numpy as np
import torch.nn as nn

def mixup_data(x, y, alpha=0.2):
    # Генерируем случайный коэффициент смешивания из Beta-распределения
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
        
    batch_size = x.size()[0]
    # Перемешиваем индексы батча
    index = torch.randperm(batch_size).to(x.device)

    # Смешиваем картинки (тензоры)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    # Сохраняем оригинальные и перемешанные метки
    y_a, y_b = y, y[index]
    
    return mixed_x, y_a, y_b, lam

# --- Использование в цикле обучения ---
criterion = nn.CrossEntropyLoss()

for inputs, labels in dataloader:
    inputs, labels = inputs.to(device), labels.to(device)
    
    # Применяем Mixup
    mixed_inputs, targets_a, targets_b, lam = mixup_data(inputs, labels, alpha=0.4)
    
    outputs = model(mixed_inputs)
    
    # Считаем смешанный лосс!
    loss = lam * criterion(outputs, targets_a) + (1 - lam) * criterion(outputs, targets_b)
    
    loss.backward()
    optimizer.step()

# **Трик 3: Label Smoothing (Сглаживание меток)**
Зачем: Нейросети часто бывают слишком самоуверенны. Если модель выдает вероятность 99.9% для одного класса, она перестает учиться. Более того, если в данных жюри есть ошибки разметки (а они почти всегда есть), излишняя уверенность убьет твой скор.
Суть: Мы говорим модели: "Правильный ответ — класс 1, но не будь уверена на 100%. Будь уверена на 90%, а 10% размажь по остальным классам".

В современном PyTorch это делается добавлением ровно одного аргумента:

In [ ]:
import torch.nn as nn

# label_smoothing=0.1 означает, что правильному классу отдается 90% вероятности,
# а оставшиеся 10% поровну делятся между неверными классами.
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Всё! Дальше используешь этот criterion как обычно. 
# Это дает бесплатный буст +1-2% к accuracy на грязных датасетах.

# **Трик 4: Stochastic Weight Averaging (SWA)**
Зачем: Это легальный чит. Вместо того чтобы обучать 3 разные модели для ансамбля (на что часто нет времени), мы усредняем "веса" одной и той же модели на последних эпохах обучения. Это позволяет модели найти более "плоский" и надежный минимум ошибки.
Суть: В PyTorch есть встроенный модуль для этого. Мы просто сохраняем копию модели и на последних эпохах вливаем в нее новые веса.

In [ ]:
from torch.optim.swa_utils import AveragedModel, SWALR
from torch.optim.lr_scheduler import CosineAnnealingLR

# 1. Твоя обычная модель
model = AutoModelForSequenceClassification.from_pretrained("bert-base").to(device)

# 2. Создаем SWA копию (она сама будет усреднять веса под капотом)
swa_model = AveragedModel(model)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

# Обычный шедулер (например, на первые 7 эпох)
scheduler = CosineAnnealingLR(optimizer, T_max=7)
# Специальный шедулер для SWA (на последние 3 эпохи он держит LR высоким)
swa_scheduler = SWALR(optimizer, swa_lr=5e-5)

epochs = 10
swa_start = 7 # Начинаем усреднять веса с 7-й эпохи

for epoch in range(epochs):
    model.train()
    for inputs, labels in dataloader:
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    
    # Логика шедулеров и SWA
    if epoch >= swa_start:
        swa_model.update_parameters(model) # Вливаем текущие веса в SWA-модель
        swa_scheduler.step()
    else:
        scheduler.step()

# КРИТИЧНО ПЕРЕД ИНФЕРЕНСОМ: 
# У SWA-модели нужно обновить BatchNorm статистику (если это CV модель вроде ResNet)
# Для трансформеров (LayerNorm) это не обязательно, но безопасно вызвать для любой модели:
torch.optim.swa_utils.update_bn(dataloader, swa_model)

# Инференс делаем через swa_model!
swa_model.eval()
preds = swa_model(test_inputs)

# **1. Для моделей-Энкодеров (BERT, RoBERTa, DeBERTa) -> Mean Pooling**

Это золотой стандарт для задач семантического поиска, сравнения текстов и контрастивного обучения. Вместо того чтобы полагаться на один токен в начале предложения, мы усредняем эмбеддинги всех реальных слов, игнорируя пустышки ([PAD]).

Вот бронебойная функция, которую используют все дата-саентисты. Скопируй её и используй вместо [:, 0, :].

In [ ]:
import torch

def mean_pooling(model_output, attention_mask):
    # 1. Достаем все векторы всех токенов (размер: [Batch, Seq_Len, Hidden_Dim])
    token_embeddings = model_output.last_hidden_state 
    
    # 2. Растягиваем маску внимания, чтобы она совпадала по размеру с эмбеддингами
    # mask изначально [Batch, Seq_Len]. Делаем ее [Batch, Seq_Len, Hidden_Dim]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    
    # 3. Умножаем векторы на маску. 
    # Реальные слова умножаются на 1 (остаются собой). 
    # [PAD] токены умножаются на 0 (становятся нулями и не портят среднее).
    token_embeddings_masked = token_embeddings * input_mask_expanded
    
    # 4. Суммируем векторы вдоль оси слов (dim=1)
    sum_embeddings = torch.sum(token_embeddings_masked, 1)
    
    # 5. Считаем, сколько было реальных слов, чтобы поделить на это число
    # clamp(min=1e-9) защищает от деления на ноль, если вдруг маска вся пустая
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    
    # 6. Получаем итоговый вектор (Сумма / Количество)
    mean_pooled = sum_embeddings / sum_mask
    
    return mean_pooled

# --- Использование ---
# outputs = model(**inputs)
# sentence_embedding = mean_pooling(outputs, inputs['attention_mask'])

# **2. Для современных LLM (LLaMA, Qwen, Mistral) -> Last Token Pooling**

У генеративных моделей (Декодеров) вообще нет [CLS] токена! У них архитектура устроена так, что каждое слово "видит" только те слова, которые стоят слева от него.
Значит, самое последнее слово в предложении накопило в себе смысл всего предыдущего текста.

Вот как достать вектор из LLM:

In [ ]:
def last_token_pooling(model_output, attention_mask):
    # Достаем все векторы
    token_embeddings = model_output.last_hidden_state
    
    # Находим индексы последних реальных слов для каждого текста в батче.
    # Суммируем маску (сколько единиц) и вычитаем 1 (т.к. индексы с нуля).
    sequence_lengths = attention_mask.sum(dim=1) - 1
    
    batch_size = token_embeddings.shape[0]
    
    # Хитрая индексация: берем для каждого элемента в батче его последний токен
    last_token_embeddings = token_embeddings[torch.arange(batch_size, device=token_embeddings.device), sequence_lengths]
    
    return last_token_embeddings

# --- Использование ---
# outputs = model(**inputs)
# llm_embedding = last_token_pooling(outputs, inputs['attention_mask'])


**Часть 1. Cosine Similarity (Косинусное сходство)
Косинусное сходство показывает, насколько два вектора смотрят в одном направлении, независимо от их длины.**
Значения лежат в диапазоне от -1 (смотрят в противоположные стороны) до 1 (идентичные направления). В ML векторы эмбеддингов редко бывают отрицательными (особенно после ReLU), поэтому на практике сходство обычно от 0 до 1.

В списке разрешенных библиотек у тебя есть scikit-learn и torch. Вот как считать матрицу сходства "всех со всеми" (Pairwise Similarity):

In [ ]:
import torch
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import torch.nn.functional as F

# Допустим, у нас есть 3 вектора группы А и 4 вектора группы B (размерность 512)
A_np = np.random.rand(3, 512)
B_np = np.random.rand(4, 512)

# === ВАРИАНТ 1: Через scikit-learn (Просто и надежно для numpy) ===
# На выходе матрица размера (3, 4)
# sim_matrix[i, j] - это сходство i-го вектора из А и j-го вектора из B
sim_matrix_sklearn = cosine_similarity(A_np, B_np) 


# === ВАРИАНТ 2: Через PyTorch (Быстро, работает на GPU) ===
A_pt = torch.tensor(A_np).cuda()
B_pt = torch.tensor(B_np).cuda()

# Нормализуем векторы (длина каждого = 1)
A_norm = F.normalize(A_pt, p=2, dim=1)
B_norm = F.normalize(B_pt, p=2, dim=1)

# Умножаем матрицу А на транспонированную матрицу B
# Получаем ту же матрицу размера (3, 4)
sim_matrix_torch = torch.matmul(A_norm, B_norm.T)

Часть 2. linear_sum_assignment (Венгерский алгоритм)

Это алгоритм из библиотеки scipy, который решает задачу о назначениях за время O(N^3).

Суть задачи: У тебя есть N рабочих и M задач. У каждого рабочего есть "стоимость" (cost) выполнения каждой задачи. Тебе нужно назначить рабочих на задачи так, чтобы минимизировать суммарную стоимость, причем каждый рабочий может взять только одну задачу, и каждая задача может быть отдана только одному рабочему.

Важное правило: Алгоритм всегда ищет МИНИМУМ (минимальную стоимость/ошибку).

In [ ]:
from scipy.optimize import linear_sum_assignment
import numpy as np

# Строки - это Рабочие (3 человека)
# Столбцы - это Задачи (3 задачи)
# Значения - это стоимость (в долларах, например), за которую рабочий сделает задачу
cost_matrix = np.array([
    [10, 50, 15], # 1-й рабочий просит за 1-ю задачу 10$, за 2-ю 50$, за 3-ю 15$
    [80, 20, 30], # 2-й рабочий...
    [40, 60, 12]  # 3-й рабочий...
])

# Запускаем Венгерский алгоритм
row_indices, col_indices = linear_sum_assignment(cost_matrix)

print("Оптимальное назначение:")
total_cost = 0
for row, col in zip(row_indices, col_indices):
    cost = cost_matrix[row, col]
    total_cost += cost
    print(f"Рабочий {row} берет Задачу {col} (Стоимость: {cost})")

print(f"Минимальная итоговая стоимость: {total_cost}")
# Рабочий 0 берет Задачу 0 (10)
# Рабочий 1 берет Задачу 1 (20)
# Рабочий 2 берет Задачу 2 (12)
# Итог: 42 (любая другая комбинация выйдет дороже!)

Часть 3. Ультимативная связка (Олимпиадный код)

Теперь представим задачу: нейросеть (OpenCLIP) выдала тебе векторы для 100 найденных на картинке лиц и векторы для 100 имен из базы данных. Тебе нужно создать идеальные уникальные пары "Лицо - Имя".

Проблема: Cosine Similarity означает "чем больше, тем лучше" (мы хотим максимальное сходство). А linear_sum_assignment ищет "чем меньше, тем лучше" (минимальную стоимость).

Решение: Мы превращаем матрицу Сходства в матрицу Стоимости (Distance/Cost) по формуле:

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy.optimize import linear_sum_assignment

def match_embeddings(embeddings_A, embeddings_B):
    """
    Принимает два массива эмбеддингов.
    Возвращает индексы идеальных пар.
    """
    # 1. Считаем матрицу косинусного сходства (чем больше число, тем больше похожи)
    # Размер: [Кол-во элементов A, Кол-во элементов B]
    similarity_matrix = cosine_similarity(embeddings_A, embeddings_B)
    
    # 2. Инвертируем сходство в "Стоимость" (Cost/Distance)
    # Теперь чем меньше число, тем ближе векторы друг к другу.
    cost_matrix = 1.0 - similarity_matrix
    
    # 3. Натравливаем Венгерский алгоритм на матрицу стоимости
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    
    # row_ind - индексы элементов из группы A
    # col_ind - соответствующие им индексы из группы B
    
    return row_ind, col_ind, similarity_matrix

# ==========================================
# ПРИМЕР ИСПОЛЬЗОВАНИЯ
# ==========================================
# Представим, что мы получили эти векторы из модели
faces_features = np.random.rand(5, 512) # 5 лиц
names_features = np.random.rand(8, 512) # 8 имен в базе (имён больше, чем лиц!)

row_indices, col_indices, sim_matrix = match_embeddings(faces_features, names_features)

print("--- Результаты идеального мэтчинга ---")
for r, c in zip(row_indices, col_indices):
    sim_score = sim_matrix[r, c]
    print(f"Лицо {r} идеально совпадает с Именем {c} (Сходство: {sim_score:.4f})")

Олимпиадный Трик на миллион:
Обрати внимание на пример: у нас 5 лиц и 8 имен. Матрица не квадратная (5х8). linear_sum_assignment работает с прямоугольными матрицами! Он сам поймет, что 3 имени лишние, и выберет только те 5 пар, которые в сумме дают абсолютный оптимум. Ни один элемент из левой группы не выберет один и тот же элемент из правой.